# Data Preprocessor — Breast Cancer Wisconsin Dataset

This notebook handles all data preprocessing in one place:
- Loading from UCI ML Repository
- Data exploration and cleaning
- Label encoding
- Train/Test split (80/20, stratified)
- Feature scaling (MinMaxScaler)
- Saving processed data via pickle

**Both STD_RF.ipynb and DP_RF.ipynb should load data from this pipeline.**

### Execution Order:
1. Run this notebook **first**
2. Run `STD_RF.ipynb` second (loads from pickle, saves baseline)
3. Run `DP_RF.ipynb` third (loads from pickle + baseline)

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import pickle
import os

## 2. Load Dataset from UCI ML Repository

In [2]:
from ucimlrepo import fetch_ucirepo

# Fetch dataset
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17)

# Data (as pandas dataframes)
X = breast_cancer_wisconsin_diagnostic.data.features
y = breast_cancer_wisconsin_diagnostic.data.targets

print("Dataset loaded successfully!")
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Dataset loaded successfully!
Features shape: (569, 30)
Target shape: (569, 1)


## 3. Data Exploration

In [3]:
# Display first few rows of features
print("Features (X):")
X.head()

Features (X):


,radius1,texture1,perimeter1,area1,smoothness1,compactness1,concavity1,concave_points1,symmetry1,fractal_dimension1,...,radius3,texture3,perimeter3,area3,smoothness3,compactness3,concavity3,concave_points3,symmetry3,fractal_dimension3
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [4]:
# Feature statistics (min/max useful for DP bounds later)
print("Feature Min/Max:")
X.describe().T[['min', 'max']]

Feature Min/Max:


,min,max
radius1,6.981000,28.11000
texture1,9.710000,39.28000
perimeter1,43.790000,188.50000
area1,143.500000,2501.00000
smoothness1,0.052630,0.16340
compactness1,0.019380,0.34540
concavity1,0.000000,0.42680
concave_points1,0.000000,0.20120
symmetry1,0.106000,0.30400
fractal_dimension1,0.049960,0.09744


In [5]:
# Display target distribution
print("Target (y):")
print(y.value_counts())

Target (y):
Diagnosis
B            357
M            212
Name: count, dtype: int64


In [6]:
# Check for missing values
print("Missing values in features:", X.isnull().sum().sum())
print("Missing values in target:", y.isnull().sum().sum())

Missing values in features: 0
Missing values in target: 0


In [7]:
# Dataset info
print("Dataset Info:")
X.info()

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   radius1             569 non-null    float64
 1   texture1            569 non-null    float64
 2   perimeter1          569 non-null    float64
 3   area1               569 non-null    float64
 4   smoothness1         569 non-null    float64
 5   compactness1        569 non-null    float64
 6   concavity1          569 non-null    float64
 7   concave_points1     569 non-null    float64
 8   symmetry1           569 non-null    float64
 9   fractal_dimension1  569 non-null    float64
 10  radius2             569 non-null    float64
 11  texture2            569 non-null    float64
 12  perimeter2          569 non-null    float64
 13  area2               569 non-null    float64
 14  smoothness2         569 non-null    float64
 15  compactness2        569 non-null    float64
 16  conca

## 4. Data Preprocessing

### 4.1 Encode Target Variable

In [8]:
# Encode target variable (M = Malignant = 1, B = Benign = 0)
le = LabelEncoder()
y_encoded = le.fit_transform(y.values.ravel())

print("Label encoding:")
print(f"Classes: {le.classes_}")
print(f"Encoded values: {np.unique(y_encoded)}")

Label encoding:
Classes: ['B' 'M']
Encoded values: [0 1]


### 4.2 Train/Test Split

In [9]:
# Split data into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded  # Maintain class distribution
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")
print(f"\nTraining Target Distribution:")
print(pd.Series(y_train).value_counts())
print(f"\nTesting Target Distribution:")
print(pd.Series(y_test).value_counts())

Training set size: 455
Testing set size: 114

Training Target Distribution:
0    285
1    170
Name: count, dtype: int64

Testing Target Distribution:
0    72
1    42
Name: count, dtype: int64


### 4.3 Feature Scaling (MinMaxScaler)

**Critical for Differential Privacy**: MinMaxScaler bounds all features to [-1, 1], providing:
1. **Guaranteed theoretical bounds** — no privacy leakage from data-dependent bound computation
2. **Known max L₂ norm** — for d features, max ||x||₂ = √d
3. **Straightforward sensitivity calibration** — noise scale directly computable from bounds

In [10]:
# Scale features using MinMaxScaler with range [-1, 1]
# IMPORTANT: fit on training data ONLY, then transform both
scaler = MinMaxScaler(feature_range=(-1, 1))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

n_features = X_train_scaled.shape[1]

print("Feature Scaling Complete!")
print(f"\nTraining Set Shape: {X_train_scaled.shape}")
print(f"Test Set Shape: {X_test_scaled.shape}")

# Verify scaling bounds
print(f"\nScaled training data statistics:")
print(f"Min: {X_train_scaled.min(axis=0)}")
print(f"Max: {X_train_scaled.max(axis=0)}")
print(f"\nAll features bounded to [-1, 1]: {np.allclose(X_train_scaled.min(axis=0), -1.0) and np.allclose(X_train_scaled.max(axis=0), 1.0)}")
print(f"Max L2 norm (theoretical): sqrt({n_features}) = {np.sqrt(n_features):.4f}")
print(f"Max L2 norm (actual train): {np.max(np.linalg.norm(X_train_scaled, axis=1)):.4f}")

Feature Scaling Complete!

Training Set Shape: (455, 30)
Test Set Shape: (114, 30)

Scaled training data statistics:
Min: [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1.
 -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
Max: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1.]

All features bounded to [-1, 1]: True
Max L2 norm (theoretical): sqrt(30) = 5.4772
Max L2 norm (actual train): 4.7253


## 5. Define Theoretical Bounds for DP Models

With MinMaxScaler(feature_range=(-1, 1)), bounds are **data-independent** and derived purely from the scaler's output guarantee. No privacy leakage — these are mathematical properties of the transformation, not statistics of the data.

In [11]:
# Theoretical bounds from MinMaxScaler: every feature is in [-1, 1]
# These bounds are data-independent — they follow from the scaler definition
n_features = X_train_scaled.shape[1]
lower_bound = np.full(n_features, -1.0)
upper_bound = np.full(n_features, 1.0)
bounds = (lower_bound, upper_bound)

print(f"Theoretical feature bounds (data-independent):")
print(f"  Lower: [-1.0] × {n_features} features")
print(f"  Upper: [+1.0] × {n_features} features")
print(f"\nDP Sensitivity properties:")
print(f"  Max L2 norm of any sample: sqrt({n_features}) = {np.sqrt(n_features):.4f}")
print(f"  Per-feature range: 2.0 (from -1 to +1)")
print(f"\nNo privacy leakage: bounds are a mathematical property of MinMaxScaler, not data statistics.")

# Create data directory if it doesn't exist
data_dir = './data'
os.makedirs(data_dir, exist_ok=True)

# Save processed data using pickle
data = {
    'X_train': X_train_scaled,
    'X_test': X_test_scaled,
    'y_train': y_train,
    'y_test': y_test,
    'feature_names': X.columns.tolist(),
    'bounds': bounds,
    'scaler': scaler,
    'label_encoder': le
}

with open(f'{data_dir}/processed_data.pkl', 'wb') as f:
    pickle.dump(data, f)

print(f"\n✓ Data saved to {data_dir}/processed_data.pkl")
print(f"\nSaved components:")
for key in data.keys():
    if isinstance(data[key], np.ndarray):
        print(f"  - {key}: {data[key].shape}")
    else:
        print(f"  - {key}: {type(data[key]).__name__}")

Theoretical feature bounds (data-independent):
  Lower: [-1.0] × 30 features
  Upper: [+1.0] × 30 features

DP Sensitivity properties:
  Max L2 norm of any sample: sqrt(30) = 5.4772
  Per-feature range: 2.0 (from -1 to +1)

No privacy leakage: bounds are a mathematical property of MinMaxScaler, not data statistics.

✓ Data saved to ./data/processed_data.pkl

Saved components:
  - X_train: (455, 30)
  - X_test: (114, 30)
  - y_train: (455,)
  - y_test: (114,)
  - feature_names: list
  - bounds: tuple
  - scaler: MinMaxScaler
  - label_encoder: LabelEncoder


## 6. Verify Saved Data

In [12]:
# Verify saved data
with open(f'{data_dir}/processed_data.pkl', 'rb') as f:
    loaded_data = pickle.load(f)

print("Data verification successful!")
print(f"X_train shape: {loaded_data['X_train'].shape}")
print(f"X_test shape: {loaded_data['X_test'].shape}")
print(f"y_train shape: {loaded_data['y_train'].shape}")
print(f"y_test shape: {loaded_data['y_test'].shape}")
print(f"Feature names: {loaded_data['feature_names'][:5]}... ({len(loaded_data['feature_names'])} total)")
print(f"Scaler: {type(loaded_data['scaler']).__name__}")
print(f"Label Encoder: {type(loaded_data['label_encoder']).__name__}")

Data verification successful!
X_train shape: (455, 30)
X_test shape: (114, 30)
y_train shape: (455,)
y_test shape: (114,)
Feature names: ['radius1', 'texture1', 'perimeter1', 'area1', 'smoothness1']... (30 total)
Scaler: MinMaxScaler
Label Encoder: LabelEncoder
